# Phase 2: Lag Feature Engineering, Cross-Correlation & Exploratory Data Analysis (EDA)

**Project:** Predicting Dengue Outbreaks in Mumbai: A Deep Learning Approach Integrating Climatic and Epidemiological Datasets  
**Target:** Weekly Dengue Cases (`dengue_cases`) & Incidence Rate per 100k  
**Exogenous Drivers:** ERA5 Historical Climate (`Rainfall_Cumulative_mm`, `Temp_Mean_C`, `Temp_Max_C`, `Temp_Min_C`, `Relative_Humidity_Pct`)  
**Auxiliary Vector Signal:** Malaria Surveillance (`malaria_cases`)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14

## 1. Load Mumbai Master Dataset

In [ ]:
master_path = '../data/final/mumbai_dengue_master.csv'
df = pd.read_csv(master_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Master Dataset Shape: {df.shape}")
df.head()

## 2. Lag Feature Engineering (Lags 1 to 8 Weeks & Rolling Windows)

Because mosquito lifecycles (*Aedes aegypti*) and viral incubation (Extrinsic Incubation Period) take multiple weeks, climate impacts dengue cases **2 to 8 weeks after the meteorological event**.

In [ ]:
climate_vars = ['Temp_Mean_C', 'Temp_Max_C', 'Temp_Min_C', 'Rainfall_Cumulative_mm', 'Relative_Humidity_Pct']
epi_vars = ['dengue_cases', 'dengue_incidence_per_100k', 'malaria_cases', 'malaria_incidence_per_100k']

# 1. Generate Climate Lags (t-1 to t-8)
for var in climate_vars:
    for lag in range(1, 9):
        df[f"{var}_lag{lag}"] = df[var].shift(lag)
        
# 2. Autoregressive Disease Lags (t-1 to t-4)
for var in epi_vars:
    for lag in range(1, 5):
        df[f"{var}_lag{lag}"] = df[var].shift(lag)
        
# 3. Rolling Cumulative & Moving Average Features
df['Rainfall_2w_sum'] = df['Rainfall_Cumulative_mm'].rolling(window=2).sum()
df['Rainfall_4w_sum'] = df['Rainfall_Cumulative_mm'].rolling(window=4).sum()
df['Temp_Mean_2w_avg'] = df['Temp_Mean_C'].rolling(window=2).mean()
df['Temp_Mean_4w_avg'] = df['Temp_Mean_C'].rolling(window=4).mean()
df['Relative_Humidity_2w_avg'] = df['Relative_Humidity_Pct'].rolling(window=2).mean()
df['Relative_Humidity_4w_avg'] = df['Relative_Humidity_Pct'].rolling(window=4).mean()

# Drop unlagged edge rows
df_model_ready = df.dropna().reset_index(drop=True)
print(f"Enriched Model-Ready Shape: {df_model_ready.shape}")
df_model_ready.to_csv('../data/final/mumbai_dengue_model_ready.csv', index=False)
print("Saved model-ready dataset to data/final/mumbai_dengue_model_ready.csv")

## 3. Cross-Correlation & Optimal Lag Analysis

In [ ]:
lags = list(range(1, 9))
corr_results = {var: [] for var in climate_vars}

for var in climate_vars:
    for lag in lags:
        corr = df_model_ready['dengue_cases'].corr(df_model_ready[f"{var}_lag{lag}"])
        corr_results[var].append(corr)
        
corr_df = pd.DataFrame(corr_results, index=[f"Lag_{l}w" for l in lags])
display(corr_df.round(4))

# Plot Lag Curves
plt.figure(figsize=(10, 6))
for var in climate_vars:
    plt.plot(lags, corr_results[var], marker='o', lw=2, label=var)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Cross-Correlation vs Lag Horizon (1 to 8 Weeks)', fontweight='bold')
plt.xlabel('Lag (Weeks Prior)', fontweight='bold')
plt.ylabel('Pearson Correlation (r)', fontweight='bold')
plt.xticks(lags)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Feature Correlation Heatmap

In [ ]:
selected_features = [
    'dengue_cases', 'dengue_incidence_per_100k',
    'malaria_cases', 'malaria_cases_lag2',
    'Rainfall_Cumulative_mm', 'Rainfall_Cumulative_mm_lag4', 'Rainfall_Cumulative_mm_lag6', 'Rainfall_4w_sum',
    'Temp_Mean_C', 'Temp_Mean_C_lag4',
    'Relative_Humidity_Pct', 'Relative_Humidity_Pct_lag4'
]

plt.figure(figsize=(11, 9))
sns.heatmap(df_model_ready[selected_features].corr(), annot=True, fmt=".2f", cmap='coolwarm', vmin=-0.5, vmax=1.0)
plt.title('Correlation Heatmap of Dengue Targets and Key Lagged Drivers', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()